# Pipeline v8 — 07 Residuo + v6, especialistas por cluster (Labo 3 BA)

Un único modelo ensemble reproducible. Conserva la base densa, reproduce el régimen
de validación reciente de 07_Residuo_sobre_baseline y combina sus predicciones con
las ramas de variación/tendencia de v6 mediante pesos regularizados por cluster.


## 0 — Ambiente

In [1]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
import duckdb
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_EXP = BUCKET / "exp_residuo"
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET: {BUCKET}")
print(f"salida: {RUTA_EXP}")

BUCKET: /home/ds/buckets/b1
salida: /home/ds/buckets/b1/exp_residuo


## 1 — Palancas

In [2]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ── Datos ────────────────────────────────────────────────────────────
    'solo_productos_target': False,  # todos los productos con ventas, como exige la cátedra   # los 780 que se entregan
    'muestra_productos': None,       # None = todos. Un numero para probar el pipe rapido.
    'horizonte': 2,
    'max_lags': 12,

    # ── Particion (misma que el pipe, para poder comparar) ───────────────
    'meses_train': rango_meses(201701, 201710), # targets observables antes de dic-2017
    'meses_val':   [201712],               # dic-2017 -> feb-2018
    'meses_test':  [201812],               # dic-2018 -> feb-2019
    'reentrenar_con_val_para_test': True,

    # ── EL BASELINE ──────────────────────────────────────────────────────
    # Sobre que se calcula el residuo. 'auto' prueba todos y elige por VALIDACION.
    #   'tn0'      -> repetir el ultimo mes (el naive)
    #   'ma3/6/12' -> promedio movil de N meses
    #   'ma_pond'  -> 0.5*tn0 + 0.3*tn1 + 0.2*tn2, pesos fijos
    #   'lineal'   -> Ridge sobre los lags: el promedio movil con pesos APRENDIDOS.
    #                 Es la version formal de la regresion stepwise que dio 0,23.
    'baseline': 'ma_pond',

    # ── EL ESQUEMA ───────────────────────────────────────────────────────
    # 'auto' compara los seis y sigue con el mejor en validacion.
    # Forzarlo sirve para aislar un esquema y compararlo en el leaderboard.
    #   'A_baseline' | 'B_lgbm_nivel' | 'C_lineal_nivel'
    #   'D_lgbm_residuo' | 'E_lineal_mas_lgbm' | 'F_lgbm_hojas_lineales'
    'esquema': 'B_lgbm_nivel',

    # ── Optuna sobre el esquema ganador ──────────────────────────────────
    'n_trials': 50,
    'techo_arboles': 800,

    # ── Ridge ────────────────────────────────────────────────────────────
    # Regularizacion de la parte lineal. Con lags muy correlacionados entre si
    # (que es el caso) sin regularizar los coeficientes se vuelven inestables.
    'ridge_alpha': 1.0,

    # ── Entrega ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semillas_ensemble': [109903, 109927],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-ba',
    'submit': True,
    'submit_ensembles': True,  # activar sólo después de mirar el score v4 puro
    'mensaje_submit': None,

    'semilla': 109903,
    'sufijo': 'v5_clusters_tweedie_ba',
}

H = PARAM['horizonte']
L = PARAM['max_lags']

EXPERIMENTO = (f"residuo_p_{L}lags_base-{PARAM['baseline']}_esq-{PARAM['esquema']}"
               f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
               f"_test{PARAM['meses_test'][0]}"
               + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"EXPERIMENTO: {EXPERIMENTO}")
print(f"carpeta    : {DIR_OUT.relative_to(BUCKET)}")

EXPERIMENTO: residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba
carpeta    : exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba


## 2 — Panel denso de la cátedra y features

Primero se materializan todas las combinaciones válidas `cliente × producto × período`.
Si no hubo compra, las toneladas y solicitudes se completan con cero. El modelo residual
sigue trabajando agregado por producto, pero su fuente es esta base enriquecida.


In [3]:
t0 = time.time()

prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
target_ids = pl.read_csv(
    DIR_RAW / "product_id_apredecir201912.txt", separator="\t"
)["product_id"].to_list()

PATH_DENSA = DIR_OUT / "panel_cliente_producto_periodo.parquet"
PATH_RESUMEN = DIR_OUT / "resumen_panel_denso.csv"


def construir_panel_denso():
    con = duckdb.connect()
    con.execute("SET preserve_insertion_order=false")
    con.execute(f"""CREATE TABLE sellin AS
        SELECT CAST(customer_id AS INTEGER) customer_id,
               CAST(product_id AS INTEGER) product_id,
               CAST(periodo AS INTEGER) periodo,
               CAST(tn AS DOUBLE) tn,
               CAST(cust_request_tn AS DOUBLE) req_tn,
               CAST(cust_request_qty AS DOUBLE) req_qty,
               CAST(plan_precios_cuidados AS INTEGER) precios_cuidados
        FROM read_csv_auto('{DIR_RAW / 'sell-in.txt.gz'}')""")
    con.execute(f"""CREATE TABLE target AS
        SELECT CAST(product_id AS INTEGER) product_id
        FROM read_csv_auto('{DIR_RAW / 'product_id_apredecir201912.txt'}')""")

    filtro = "WHERE product_id IN (SELECT product_id FROM target)" if PARAM['solo_productos_target'] else ""
    con.execute(f"""CREATE TABLE base AS
        SELECT customer_id, product_id, periodo,
               SUM(tn) tn, SUM(req_tn) req_tn, SUM(req_qty) req_qty,
               MAX(precios_cuidados) precios_cuidados
        FROM sellin {filtro}
        GROUP BY customer_id, product_id, periodo""")

    if PARAM['muestra_productos']:
        con.execute(f"""CREATE TABLE muestra AS
            SELECT product_id FROM base GROUP BY product_id
            ORDER BY SUM(tn) DESC LIMIT {int(PARAM['muestra_productos'])}""")
        con.execute("DELETE FROM base WHERE product_id NOT IN (SELECT product_id FROM muestra)")

    con.execute("CREATE TABLE periodos AS SELECT DISTINCT periodo FROM base")
    con.execute("""CREATE TABLE vida_prod AS
        SELECT product_id, MIN(periodo) nace, MAX(periodo) muere
        FROM base GROUP BY product_id""")
    # Igual que v3: los productos de entrega continúan hasta 201912 aunque no vendan.
    con.execute("""UPDATE vida_prod SET muere=201912
        WHERE product_id IN (SELECT product_id FROM target) AND muere < 201912""")
    con.execute("""CREATE TABLE primer_cli AS
        SELECT customer_id, MIN(periodo) nace_cli
        FROM base GROUP BY customer_id""")
    con.execute("""CREATE TABLE grid AS
        SELECT c.customer_id, v.product_id, p.periodo
        FROM vida_prod v
        JOIN periodos p ON p.periodo BETWEEN v.nace AND v.muere
        CROSS JOIN primer_cli c
        WHERE p.periodo >= c.nace_cli""")

    con.execute("""CREATE TABLE densa AS
        SELECT g.customer_id, g.product_id, g.periodo,
               COALESCE(b.tn, 0.0) tn,
               COALESCE(b.req_tn, 0.0) req_tn,
               COALESCE(b.req_qty, 0.0) req_qty,
               COALESCE(b.precios_cuidados, 0) precios_cuidados
        FROM grid g LEFT JOIN base b
        USING (customer_id, product_id, periodo)""")

    resumen = con.execute("""SELECT
        COUNT(*) filas_totales,
        COUNT(DISTINCT customer_id) clientes,
        COUNT(DISTINCT product_id) productos,
        COUNT(DISTINCT periodo) periodos,
        COUNT(*) FILTER (WHERE tn > 0) filas_con_compra,
        COUNT(*) FILTER (WHERE tn = 0) filas_en_cero
        FROM densa""").pl()

    tmp = str(PATH_DENSA) + ".tmp"
    con.execute(f"COPY (SELECT * FROM densa ORDER BY customer_id, product_id, periodo) "
                f"TO '{tmp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    Path(tmp).replace(PATH_DENSA)
    con.close()
    resumen.write_csv(PATH_RESUMEN)
    return resumen


if PATH_DENSA.exists():
    print("[panel denso] RESUME")
    if PATH_RESUMEN.exists():
        resumen_denso = pl.read_csv(PATH_RESUMEN)
    else:
        densa_lazy = pl.scan_parquet(PATH_DENSA)
        resumen_denso = densa_lazy.select(
            pl.len().alias("filas_totales"),
            pl.col("customer_id").n_unique().alias("clientes"),
            pl.col("product_id").n_unique().alias("productos"),
            pl.col("periodo").n_unique().alias("periodos"),
            (pl.col("tn") > 0).sum().alias("filas_con_compra"),
            (pl.col("tn") == 0).sum().alias("filas_en_cero"),
        ).collect()
        resumen_denso.write_csv(PATH_RESUMEN)
else:
    resumen_denso = construir_panel_denso()

print("\nRESUMEN DEL PANEL DENSO EXIGIDO POR LA CATEDRA")
print(resumen_denso)
print(f"checkpoint: {PATH_DENSA}")

# Agregación posterior a producto-mes. n_clientes cuenta compradores reales,
# no todos los clientes incorporados mediante las filas en cero.
panel = (pl.scan_parquet(PATH_DENSA)
    .group_by(["product_id", "periodo"])
    .agg(
        pl.col("tn").sum().alias("tn"),
        pl.col("req_tn").sum().alias("req_tn"),
        pl.col("req_qty").sum().alias("req_qty"),
        pl.col("customer_id").filter(pl.col("tn") > 0).n_unique().alias("n_clientes"),
        pl.col("precios_cuidados").max().alias("precios_cuidados"),
    ).collect()
    .with_columns((((pl.col("periodo") // 100) * 12)
                   + (pl.col("periodo") % 100)).alias("m")))

# La base formal anterior continúa conteniendo todos los productos. Para el modelo
# replicamos la población de 07_Residuo_sobre_baseline: sólo productos de entrega.
panel = panel.filter(pl.col("product_id").is_in(target_ids))
print(f"población del modelo: {panel['product_id'].n_unique()} productos objetivo")

vida = panel.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"),
    pl.col("m").max().alias("m_muere")
)

panel = (panel.join(vida, on="product_id", how="left")
               .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                     on="product_id", how="left")
               .with_columns(
                   pl.when(pl.col("m") >= pl.col("m_nace"))
                     .then(pl.col("m") - pl.col("m_nace")).otherwise(-1).alias("edad"))
               .sort(["product_id", "m"]))

print(f"panel agregado: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")

# ── Totales para los shares (todos del mes t: contexto, no futuro) ──────
for niv in ("cat1", "cat2", "cat3"):
    t = panel.group_by([niv, "m"]).agg(pl.col("tn").sum().alias(f"tn_{niv}"))
    panel = panel.join(t, on=[niv, "m"], how="left")
mercado = panel.group_by("m").agg(pl.col("tn").sum().alias("tn_mercado"))
panel = panel.join(mercado, on="m", how="left")


def div_segura(num, den, nombre):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then(pl.col(num) / pl.col(den)).otherwise(0.0).alias(nombre))


SHARES = [f"sh_{n}" for n in ("cat1", "cat2", "cat3", "mercado")]
panel = panel.with_columns([div_segura("tn", f"tn_{n}", f"sh_{n}")
                            for n in ("cat1", "cat2", "cat3", "mercado")])

# ── Lags, promedios móviles, deltas e índices ───────────────────────────
df = panel.sort(["product_id", "m"]).with_columns(
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over("product_id").alias(f"tn_ma{w}") for w in (3, 6, 12)],
    *[pl.col(s).shift(k).over("product_id").alias(f"{s}_lag{k}")
      for s in SHARES for k in (1, 2, 3)],
    *[pl.col(s).rolling_mean(3).over("product_id").alias(f"{s}_ma3") for s in SHARES],
    pl.col("n_clientes").shift(1).over("product_id").alias("n_clientes_lag1"),
    pl.col("n_clientes").rolling_mean(3).over("product_id").alias("n_clientes_ma3"),
    pl.col("req_qty").shift(1).over("product_id").alias("qty_lag1"),
    pl.col("tn").cum_max().over("product_id").alias("tn_pico_hasta_aca"),
    (pl.col("tn") > 0).cast(pl.Int8).alias("vendio"),
)

df = df.with_columns(
    *[(pl.col(s) - pl.col(f"{s}_lag1")).alias(f"{s}_d1") for s in SHARES],
    *[(pl.col(s) - pl.col(f"{s}_ma3")).alias(f"{s}_dma3") for s in SHARES],
    (pl.col("tn") - pl.col("tn_ma3")).alias("tn_dma3"),
    pl.col("vendio").rolling_mean(6).over("product_id").alias("frac_venta_6"),
    (pl.col("periodo") % 100).alias("mes_del_anio"),
    (pl.col("edad").is_between(0, 6)).cast(pl.Int8).alias("es_nuevo"),
)


def indice(num, den, nombre, techo=10.0):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then((pl.col(num) / pl.col(den)).clip(0.0, techo))
              .otherwise(pl.lit(None, dtype=pl.Float64)).alias(nombre))


df = df.with_columns(
    indice("tn", "tn_lag1", "idx_tn_mom"),
    indice("tn", "tn_ma3", "idx_tn_vs_ma3"),
    indice("tn", "tn_pico_hasta_aca", "idx_vs_pico"),
    indice("n_clientes", "n_clientes_lag1", "idx_clientes_mom"),
    indice("req_qty", "qty_lag1", "idx_qty_mom"),
)

# ── Target ──────────────────────────────────────────────────────────────
df = df.sort(["product_id", "m"]).with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("clase_tn"),
    ((((pl.col("m") + H - 1) // 12) * 100) + ((pl.col("m") + H - 1) % 12) + 1)
      .alias("periodo_objetivo"),
)

CATS = ["cat1", "cat2", "cat3", "brand"]
df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                      for c in CATS])

NO_FEAT = {"product_id", "periodo", "m", "m_nace", "m_muere", "clase_tn",
           "periodo_objetivo"}
FEATURES = [c for c in df.columns if c not in NO_FEAT]

print(f"features: {len(FEATURES)}   filas: {df.height:,}   [{time.time()-t0:.0f}s]")
print(f"con target: {int(df['clase_tn'].is_not_null().sum()):,}")


[panel denso] RESUME

RESUMEN DEL PANEL DENSO EXIGIDO POR LA CATEDRA
shape: (1, 6)
┌───────────────┬──────────┬───────────┬──────────┬──────────────────┬───────────────┐
│ filas_totales ┆ clientes ┆ productos ┆ periodos ┆ filas_con_compra ┆ filas_en_cero │
│ ---           ┆ ---      ┆ ---       ┆ ---      ┆ ---              ┆ ---           │
│ i64           ┆ i64      ┆ i64       ┆ i64      ┆ i64              ┆ i64           │
╞═══════════════╪══════════╪═══════════╪══════════╪══════════════════╪═══════════════╡
│ 17173448      ┆ 597      ┆ 1233      ┆ 36       ┆ 2945818          ┆ 14227630      │
└───────────────┴──────────┴───────────┴──────────┴──────────────────┴───────────────┘
checkpoint: /home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba/panel_cliente_producto_periodo.parquet
población del modelo: 780 productos objetivo
panel agregado: 22,375 filas · 780 productos
features: 72   filas: 22,375   [3s]
c

## 3 — Partición y control de leakage

In [4]:
sup = df.filter(pl.col("clase_tn").is_not_null())
periodos_sup = sorted(sup["periodo"].unique().to_list())
MESES_TRAIN = [m for m in PARAM['meses_train'] if m in periodos_sup]
MESES_VAL   = [m for m in PARAM['meses_val'] if m in periodos_sup]
MESES_TEST  = [m for m in PARAM['meses_test'] if m in periodos_sup]
MESES_INFER = sorted(df.filter(pl.col("clase_tn").is_null())["periodo"].unique().to_list())[-H:]
infer = df.filter(pl.col("periodo").is_in(MESES_INFER))

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


def a_m(p):
    return (p // 100) * 12 + (p % 100)


print("CONTROL DE LEAKAGE")
print("=" * 74)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                     (MESES_VAL, MESES_TEST, "val", "test")):
    g = a_m(min(b)) - a_m(max(a))
    chk(g >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {g} >= horizonte {H}")
if PARAM['reentrenar_con_val_para_test']:
    g = a_m(min(MESES_TEST)) - a_m(max(MESES_TRAIN + MESES_VAL))
    chk(g >= H, f"gap (train+val) -> test = {g} >= {H}")
chk(max(MESES_TRAIN) < min(MESES_VAL) and max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
chk("m_muere" not in FEATURES, "m_muere (dato del futuro) no es feature")
chk(not (set(FEATURES) & {"clase_tn", "periodo_objetivo"}), "el target no es feature")

# el shift del target, verificado fila por fila en la serie mas larga
_u = sup.group_by("product_id").agg(pl.len().alias("n")).sort("n", descending=True).head(1)
_s = df.filter(pl.col("product_id") == _u["product_id"][0]).sort("m")
_tn, _cl = _s["tn"].to_list(), _s["clase_tn"].to_list()
_mal = [i for i in range(len(_tn) - H)
        if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
chk(not _mal, f"clase_tn[i] == tn[i+{H}] en el producto {_u['product_id'][0]} "
              f"({len(_tn)} meses, {len(_mal)} discrepancias)")

print("=" * 74)
if errores:
    raise RuntimeError(f"Leakage: {errores}")
print(f"TRAIN {len(MESES_TRAIN)} meses ({sup.filter(pl.col('periodo').is_in(MESES_TRAIN)).height:,} filas)"
      f" · VAL {MESES_VAL} · TEST {MESES_TEST} · INFER {MESES_INFER}")

CONTROL DE LEAKAGE
  [ok   ] gap train(201710) -> val(201712) = 2 >= horizonte 2
  [ok   ] gap val(201712) -> test(201812) = 12 >= horizonte 2
  [ok   ] gap (train+val) -> test = 12 >= 2
  [ok   ] orden cronologico train < val < test
  [ok   ] m_muere (dato del futuro) no es feature
  [ok   ] el target no es feature
  [ok   ] clase_tn[i] == tn[i+2] en el producto 20001 (36 meses, 0 discrepancias)
TRAIN 10 meses (5,159 filas) · VAL [201712] · TEST [201812] · INFER [201911, 201912]


## 4 — El WAPE y los baselines

El baseline es lo que se le regala al modelo: la estimación del **nivel**, que ya
funciona. El modelo después sólo tiene que aprender la **desviación**.

El más interesante es `lineal`: una Ridge sobre los lags, o sea **un promedio móvil con
los pesos aprendidos en vez de fijos**. Es la versión formal de la regresión stepwise que
dio 0,23, y por eso está acá como candidato de primera clase y no como curiosidad.

Ridge y no mínimos cuadrados puros porque los lags están muy correlacionados entre sí
(`tn_lag1` y `tn_lag2` se parecen mucho): sin regularizar, los coeficientes se vuelven
grandes y de signos alternados, y el modelo deja de generalizar.

In [5]:
def wape(y_real, y_pred, ids=None) -> float:
    """WAPE en toneladas, agregando por producto. Identico al del pipe."""
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if ids is not None:
        _, inv = np.unique(np.asarray(ids), return_inverse=True)
        yr, yp = np.bincount(inv, weights=yr), np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def bloque(meses):
    return sup.filter(pl.col("periodo").is_in(meses))


tr, va, te = bloque(MESES_TRAIN), bloque(MESES_VAL), bloque(MESES_TEST)
COLS_LIN = ["tn"] + [f"tn_lag{k}" for k in range(1, L + 1)] + ["tn_ma3", "tn_ma6"]


def X_lin(b):
    """Matriz para la parte lineal: los niveles recientes, con los nulos del arranque en 0."""
    return b.select(COLS_LIN).fill_null(0.0).to_numpy()


def wape_de(b, pred):
    return wape(b["clase_tn"].to_numpy(), pred, b["product_id"].to_numpy())


# ── Los baselines ────────────────────────────────────────────────────────
def baseline_fijo(b, cual):
    if cual == "tn0":
        return b["tn"].to_numpy().astype(np.float64)
    if cual == "ma_pond":
        return (0.5 * b["tn"].fill_null(0).to_numpy()
                + 0.3 * b["tn_lag1"].fill_null(0).to_numpy()
                + 0.2 * b["tn_lag2"].fill_null(0).to_numpy())
    return b[f"tn_{cual}"].fill_null(0.0).to_numpy().astype(np.float64)


_ridge_base = Ridge(alpha=PARAM['ridge_alpha'])
_ridge_base.fit(X_lin(tr), tr["clase_tn"].to_numpy())


def baseline_de(b, cual, ridge=None):
    """Prediccion del baseline para el bloque b. Siempre >= 0."""
    if cual == "lineal":
        r = ridge if ridge is not None else _ridge_base
        return np.maximum(r.predict(X_lin(b)), 0.0)
    return np.maximum(baseline_fijo(b, cual), 0.0)


CANDIDATOS = ["tn0", "ma3", "ma6", "ma12", "ma_pond", "lineal"]
print(f"{'baseline':10s} {'WAPE val':>10s}")
print("-" * 22)
wape_base = {}
for c in CANDIDATOS:
    wape_base[c] = wape_de(va, baseline_de(va, c))
    print(f"{c:10s} {wape_base[c]:10.5f}")

BASELINE = (PARAM['baseline'] if PARAM['baseline'] != 'auto'
            else min(wape_base, key=wape_base.get))
print(f"\nbaseline elegido: {BASELINE}"
      + ("  (por validacion)" if PARAM['baseline'] == 'auto' else "  (forzado)"))

# ── Los pesos que aprendio la Ridge: el promedio movil optimizado ────────
_co = dict(zip(COLS_LIN, _ridge_base.coef_))
print(f"\npesos de la Ridge (intercepto {_ridge_base.intercept_:+.3f}):")
for k, v in sorted(_co.items(), key=lambda kv: -abs(kv[1]))[:8]:
    print(f"   {k:10s} {v:+.4f}")
print("Si los pesos son positivos y decrecientes, la Ridge encontro sola un promedio")
print("movil ponderado. Si alternan de signo, esta capturando reversion a la media.")

baseline     WAPE val
----------------------
tn0           0.22584
ma3           0.35063
ma6           0.33896
ma12          0.30352
ma_pond       0.29152
lineal        0.44090

baseline elegido: ma_pond  (forzado)

pesos de la Ridge (intercepto +0.493):
   tn         +1.0305
   tn_ma3     -0.6111
   tn_lag6    +0.4854
   tn_lag8    -0.4359
   tn_lag5    +0.4174
   tn_ma6     -0.3689
   tn_lag2    +0.3528
   tn_lag9    -0.2862
Si los pesos son positivos y decrecientes, la Ridge encontro sola un promedio
movil ponderado. Si alternan de signo, esta capturando reversion a la media.


## 5 — Los seis esquemas

Todos con las mismas features, la misma partición y la misma semilla. La única
diferencia es **quién se queda con qué parte del problema**.

El WAPE se mide siempre sobre **toneladas reconstruidas**, así que los seis números son
directamente comparables entre sí y con los del pipe.

In [6]:
PARAMS_LGBM = dict(objective="regression", metric="mae", verbosity=-1,
                   n_estimators=500, learning_rate=0.05, num_leaves=63,
                   min_child_samples=20, subsample=0.9, subsample_freq=1,
                   colsample_bytree=0.8, seed=PARAM['semilla'], n_jobs=-1,
                   deterministic=True, force_row_wise=True)


def fit_lgbm(b, target, params=None, lineal=False):
    p = dict(params or PARAMS_LGBM)
    if lineal:
        # linear_tree ajusta una REGRESION LINEAL dentro de cada hoja: le da al arbol
        # la capacidad de extrapolar que por construccion no tiene.
        p.update(linear_tree=True, linear_lambda=1.0)
    m = lgb.LGBMRegressor(**p)
    bp = b.to_pandas()
    m.fit(bp[FEATURES], bp[target].to_numpy() if hasattr(bp[target], "to_numpy") else bp[target],
          categorical_feature=CATS)
    return m


def evaluar_esquemas(meses_fit, b_eval):
    """Devuelve {esquema: pred_tn} para el bloque de evaluacion."""
    fit = bloque(meses_fit)
    base_fit = baseline_de(fit, BASELINE)
    base_ev = baseline_de(b_eval, BASELINE)
    ev = b_eval.to_pandas()
    out, modelos = {}, {}

    # A) el baseline solo
    out["A_baseline"] = base_ev

    # B) LightGBM al nivel
    mB = fit_lgbm(fit, "clase_tn")
    out["B_lgbm_nivel"] = mB.predict(ev[FEATURES]); modelos["B_lgbm_nivel"] = mB

    # C) lineal al nivel (Ridge sobre TODAS las features numericas, no solo los lags)
    _num = [c for c in FEATURES if c not in CATS]
    mC = Ridge(alpha=PARAM['ridge_alpha'])
    mC.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
    out["C_lineal_nivel"] = mC.predict(b_eval.select(_num).fill_null(0.0).to_numpy())
    modelos["C_lineal_nivel"] = mC

    # D) LightGBM al residuo del baseline
    fit_d = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_fit))
    mD = fit_lgbm(fit_d, "y_res")
    out["D_lgbm_residuo"] = base_ev + mD.predict(ev[FEATURES]); modelos["D_lgbm_residuo"] = mD

    # E) lineal para el nivel + LightGBM para lo que sobra
    #    Es la apuesta: la recta hace el nivel (extrapola), el arbol las interacciones.
    rE = Ridge(alpha=PARAM['ridge_alpha'])
    rE.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    base_lin_fit = np.maximum(rE.predict(X_lin(fit)), 0.0)
    base_lin_ev = np.maximum(rE.predict(X_lin(b_eval)), 0.0)
    fit_e = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_lin_fit))
    mE = fit_lgbm(fit_e, "y_res")
    out["E_lineal_mas_lgbm"] = base_lin_ev + mE.predict(ev[FEATURES])
    modelos["E_lineal_mas_lgbm"] = (rE, mE)

    # F) LightGBM con hojas lineales, al nivel
    mF = fit_lgbm(fit, "clase_tn", lineal=True)
    out["F_lgbm_hojas_lineales"] = mF.predict(ev[FEATURES]); modelos["F_lgbm_hojas_lineales"] = mF

    return out, modelos


t0 = time.time()
pred_val, mod_val = evaluar_esquemas(MESES_TRAIN, va)
print(f"[{time.time()-t0:.0f}s]\n")

ESQUEMAS = list(pred_val)
print(f"{'esquema':24s} {'WAPE val':>10s}")
print("-" * 36)
wape_val = {}
for e in ESQUEMAS:
    wape_val[e] = wape_de(va, pred_val[e])
    print(f"{e:24s} {wape_val[e]:10.5f}")

ESQUEMA = (PARAM['esquema'] if PARAM['esquema'] != 'auto'
           else min(wape_val, key=wape_val.get))
print(f"\nesquema utilizado: {ESQUEMA}"
      + ("  (por validacion)" if PARAM['esquema'] == 'auto' else "  (forzado)"))
_mej = 100 * (wape_val['A_baseline'] - wape_val[ESQUEMA]) / wape_val['A_baseline']
print(f"mejora sobre el baseline solo: {_mej:+.1f}%")

[12s]

esquema                    WAPE val
------------------------------------
A_baseline                  0.29152
B_lgbm_nivel                0.32699
C_lineal_nivel              0.45910
D_lgbm_residuo              0.30532
E_lineal_mas_lgbm           0.34542
F_lgbm_hojas_lineales       0.43442

esquema utilizado: B_lgbm_nivel  (forzado)
mejora sobre el baseline solo: -12.2%


## 6 — Optuna sobre el esquema ganador

Sólo se afina el esquema que ganó en validación, y **lo que se minimiza es el WAPE en
toneladas reconstruidas**, nunca el error del residuo. Si el esquema ganador es
`A_baseline` no hay nada que optimizar y esta celda lo saltea — que sería, en sí, el
resultado más interesante posible: que ningún modelo le gana al promedio.

In [7]:
def espacio(trial):
    return dict(
        objective="regression", metric="mae", verbosity=-1,
        seed=PARAM['semilla'], n_jobs=-1, subsample_freq=1,
        deterministic=True, force_row_wise=True,
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        n_estimators=trial.suggest_int("n_estimators", 200, PARAM['techo_arboles']),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )


def predecir_esquema(esquema, meses_fit, b_eval, params, semilla=None):
    """Entrena el esquema con `params` y devuelve las toneladas predichas."""
    fit = bloque(meses_fit)
    ev = b_eval.to_pandas()
    p = dict(params)
    if semilla is not None:
        p["seed"] = semilla

    if esquema == "A_baseline":
        return baseline_de(b_eval, BASELINE), None

    if esquema == "C_lineal_nivel":
        _num = [c for c in FEATURES if c not in CATS]
        r = Ridge(alpha=PARAM['ridge_alpha'])
        r.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
        return r.predict(b_eval.select(_num).fill_null(0.0).to_numpy()), r

    if esquema == "B_lgbm_nivel":
        m = fit_lgbm(fit, "clase_tn", p)
        return m.predict(ev[FEATURES]), m

    if esquema == "F_lgbm_hojas_lineales":
        m = fit_lgbm(fit, "clase_tn", p, lineal=True)
        return m.predict(ev[FEATURES]), m

    if esquema == "D_lgbm_residuo":
        bf, be = baseline_de(fit, BASELINE), baseline_de(b_eval, BASELINE)
        f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
        m = fit_lgbm(f2, "y_res", p)
        return be + m.predict(ev[FEATURES]), m

    # E_lineal_mas_lgbm
    r = Ridge(alpha=PARAM['ridge_alpha'])
    r.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    bf = np.maximum(r.predict(X_lin(fit)), 0.0)
    be = np.maximum(r.predict(X_lin(b_eval)), 0.0)
    f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
    m = fit_lgbm(f2, "y_res", p)
    return be + m.predict(ev[FEATURES]), (r, m)


SIN_HIPER = {"A_baseline", "C_lineal_nivel"}

if ESQUEMA in SIN_HIPER:
    print(f"El esquema ganador ({ESQUEMA}) no tiene hiperparametros que buscar.")
    if ESQUEMA == "A_baseline":
        print("Y eso es un resultado en si mismo: ningun modelo le gana al baseline.")
    study = None
    MEJORES = {}
else:
    study = optuna.create_study(
        direction="minimize", study_name=EXPERIMENTO,
        sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
        storage=f"sqlite:///{Path.home() / ('optuna_' + EXPERIMENTO + '.db')}",
        load_if_exists=True)

    def objective(trial):
        pred, _ = predecir_esquema(ESQUEMA, MESES_TRAIN, va, espacio(trial))
        return wape_de(va, pred)

    t0 = time.time()
    study.optimize(objective, n_trials=PARAM['n_trials'])
    MEJORES = {**espacio(optuna.trial.FixedTrial(study.best_params))}
    print(f"{len(study.trials)} trials · mejor WAPE val = {study.best_value:.5f}"
          f"  ({time.time()-t0:.0f}s)")
    print(f"mejora de Optuna sobre los defaults: "
          f"{100*(wape_val[ESQUEMA]-study.best_value)/wape_val[ESQUEMA]:+.1f}%")
    for k, v in study.best_params.items():
        print(f"   {k:22s} {v}")

50 trials · mejor WAPE val = 0.29771  (49s)
mejora de Optuna sobre los defaults: +9.0%
   num_leaves             136
   max_depth              8
   learning_rate          0.09433176265594022
   n_estimators           286
   min_child_samples      23
   subsample              0.9125472929795299
   colsample_bytree       0.9307864807790545
   reg_alpha              0.24700724500357815
   reg_lambda             3.513103697522887e-07


## 7 — Resultados: la tabla completa

`val` es optimista para el esquema ganador (Optuna lo minimizó). `test` es el número
honesto, medido una sola vez.

In [8]:
_corte_test = min(MESES_TEST)
_idx_test = (_corte_test // 100) * 12 + (_corte_test % 100) - PARAM['horizonte']
MESES_FIT_TEST = [p for p in periodos_sup
                  if ((p // 100) * 12 + (p % 100)) <= _idx_test] \
                 if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
_pars = MEJORES or PARAMS_LGBM

pred_test, _ = evaluar_esquemas(MESES_FIT_TEST, te)
if MEJORES:
    pred_test[ESQUEMA], _ = predecir_esquema(ESQUEMA, MESES_FIT_TEST, te, _pars)

print(f"{'esquema':24s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 48)
METRICAS = {}
for e in ESQUEMAS:
    wt = wape_de(te, pred_test[e])
    METRICAS[e] = {"val": wape_val[e], "test": wt}
    marca = "  <-" if e == ESQUEMA else ""
    print(f"{e:24s} {wape_val[e]:10.5f} {wt:10.5f}{marca}")

_a, _g = METRICAS['A_baseline']['test'], METRICAS[ESQUEMA]['test']
print(f"\nganador en test: {min(METRICAS, key=lambda e: METRICAS[e]['test'])}")
print(f"{ESQUEMA} vs baseline solo, en test: {100*(_a-_g)/_a:+.1f}%")
_br = METRICAS[ESQUEMA]['test'] - METRICAS[ESQUEMA]['val']
print(f"brecha test - val: {_br:+.5f}"
      + ("   <- sobreajuste a validacion" if _br > 0.02 else ""))

pl.DataFrame([{"esquema": e, **METRICAS[e]} for e in ESQUEMAS]) \
  .write_csv(DIR_OUT / "esquemas.csv")

# ── Donde gana cada uno: por volumen del producto ───────────────────────
# El WAPE pondera por volumen, asi que lo unico que mueve la aguja son los productos
# grandes. Este corte dice si la ventaja viene de ahi o de la cola.
det = te.select("product_id", "clase_tn").with_columns(
    pl.Series("base", pred_test["A_baseline"]),
    pl.Series("gana", pred_test[ESQUEMA]))
_q = det.group_by("product_id").agg(pl.col("clase_tn").sum().alias("tn"))
_cortes = _q["tn"].qcut(4, labels=["Q1 chico", "Q2", "Q3", "Q4 grande"], allow_duplicates=True)
_q = _q.with_columns(_cortes.alias("cuartil"))
det = det.join(_q.select("product_id", "cuartil"), on="product_id", how="left")

filas = []
for c in ["Q1 chico", "Q2", "Q3", "Q4 grande"]:
    b = det.filter(pl.col("cuartil") == c)
    if b.height < 5:
        continue
    filas.append({"cuartil": c, "n_filas": b.height,
                  "tn_real": round(float(b["clase_tn"].sum()), 1),
                  "wape_baseline": round(wape(b["clase_tn"], b["base"], b["product_id"]), 4),
                  f"wape_{ESQUEMA}": round(wape(b["clase_tn"], b["gana"], b["product_id"]), 4)})
por_vol = pl.DataFrame(filas)
print()
print(por_vol)
por_vol.write_csv(DIR_OUT / "por_cuartil_de_volumen.csv")
print("\nEl WAPE pondera por volumen: si la ventaja no esta en Q4, no va a mover el")
print("numero global aunque se vea grande en los cuartiles chicos.")

esquema                    WAPE val  WAPE test
------------------------------------------------
A_baseline                  0.29152    0.25136
B_lgbm_nivel                0.32699    0.31912  <-
C_lineal_nivel              0.45910    0.32204
D_lgbm_residuo              0.30532    0.34337
E_lineal_mas_lgbm           0.34542    0.27882
F_lgbm_hojas_lineales       0.43442    0.29831

ganador en test: A_baseline
B_lgbm_nivel vs baseline solo, en test: -27.0%
brecha test - val: -0.00787

shape: (4, 5)
┌───────────┬─────────┬─────────┬───────────────┬───────────────────┐
│ cuartil   ┆ n_filas ┆ tn_real ┆ wape_baseline ┆ wape_B_lgbm_nivel │
│ ---       ┆ ---     ┆ ---     ┆ ---           ┆ ---               │
│ str       ┆ i64     ┆ f64     ┆ f64           ┆ f64               │
╞═══════════╪═════════╪═════════╪═══════════════╪═══════════════════╡
│ Q1 chico  ┆ 164     ┆ 173.4   ┆ 0.3761        ┆ 1.1806            │
│ Q2        ┆ 164     ┆ 949.0   ┆ 0.3261        ┆ 0.6432            │
│ Q3     

## 8 — Entrenamiento final y entrega

Se reentrena el esquema ganador con **todos** los meses supervisados y se predice el mes
objetivo. Del modelo final no hay métrica honesta: la que se reporta es la de `test`.

In [9]:
MESES_TODOS = sorted(periodos_sup)
print(f"reentrenando {ESQUEMA} con {len(MESES_TODOS)} meses "
      f"({MESES_TODOS[0]}..{MESES_TODOS[-1]})")

preds = []
for sem in PARAM['semillas_ensemble']:
    p, _ = predecir_esquema(ESQUEMA, MESES_TODOS, infer, _pars, semilla=sem)
    preds.append(p)
    print(f"  semilla {sem} lista")
pred_infer_tn = np.maximum(np.mean(preds, axis=0), PARAM['clip_min'])

pred_infer = infer.select("product_id", "periodo", "periodo_objetivo").with_columns(
    pl.Series("tn_pred", pred_infer_tn),
    pl.Series("baseline", baseline_de(infer, BASELINE)))
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\npredicciones: {pred_infer.height:,} filas")
print(pred_infer.group_by("periodo", "periodo_objetivo").len().sort("periodo"))
print(f"\ntn_pred   min {pred_infer_tn.min():.2f}   media {pred_infer_tn.mean():.2f}   "
      f"max {pred_infer_tn.max():.2f}")
print(f"correlacion con el baseline: "
      f"{np.corrcoef(pred_infer_tn, pred_infer['baseline'].to_numpy())[0,1]:.4f}")
print("Si esa correlacion es ~1, el modelo esta repitiendo el baseline y no aporta nada.")

reentrenando B_lgbm_nivel con 34 meses (201701..201910)
  semilla 109903 lista
  semilla 109927 lista

predicciones: 1,560 filas
shape: (2, 3)
┌─────────┬──────────────────┬─────┐
│ periodo ┆ periodo_objetivo ┆ len │
│ ---     ┆ ---              ┆ --- │
│ i32     ┆ i32              ┆ u32 │
╞═════════╪══════════════════╪═════╡
│ 201911  ┆ 202001           ┆ 780 │
│ 201912  ┆ 202002           ┆ 780 │
└─────────┴──────────────────┴─────┘

tn_pred   min 0.00   media 36.27   max 1369.98
correlacion con el baseline: 0.9840
Si esa correlacion es ~1, el modelo esta repitiendo el baseline y no aporta nada.


## 9 — El CSV y el submit

In [10]:
OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")

por_producto = obj.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")
submit = oficiales.select("product_id").join(por_producto, on="product_id", how="left")
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")

print(f"mes objetivo {OBJ}: {obj.height} filas -> {por_producto.height} productos")
print(f"lista oficial: {oficiales.height}   sin prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista. Revisalo antes de subir.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10))

path_submit = DIR_OUT / f"submission_{OBJ}.csv"
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"\nGuardado: {path_submit}")


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("\nPARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    flag_submit = DIR_OUT / "submit_v4.done"
    if flag_submit.exists():
        print("\nSubmit v4 ya realizado; se evita duplicarlo.")
    kd = Path.home() / ".kaggle" / "kaggle.json"
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kd); kd.chmod(0o600); break
    if flag_submit.exists():
        pass
    elif not kd.exists():
        print("\nSin credenciales de Kaggle. El CSV ya esta generado.")
    else:
        kd.chmod(0o600)
        msg = PARAM['mensaje_submit'] or (
            f"{ESQUEMA} sobre {BASELINE} | wape_test={METRICAS[ESQUEMA]['test']:.5f}")
        ok, salida = kaggle_cli(["competitions", "submit",
                                 "-c", PARAM['kaggle_competition'],
                                 "-f", str(path_submit), "-m", msg])
        print(f"\nmensaje: {msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")
        if ok:
            flag_submit.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

mes objetivo 202002: 780 filas -> 780 productos
lista oficial: 780   sin prediccion (van en 0): 0

tn   min 0.000   media 38.187   max 1369.984   suma 29,785.8
shape: (10, 2)
┌────────────┬─────────────┐
│ product_id ┆ tn          │
│ ---        ┆ ---         │
│ i64        ┆ f64         │
╞════════════╪═════════════╡
│ 20001      ┆ 1196.712332 │
│ 20002      ┆ 1369.9844   │
│ 20003      ┆ 678.112276  │
│ 20004      ┆ 591.380943  │
│ 20005      ┆ 613.224058  │
│ 20006      ┆ 422.4535    │
│ 20007      ┆ 415.019092  │
│ 20008      ┆ 381.129426  │
│ 20009      ┆ 598.299086  │
│ 20010      ┆ 390.701027  │
└────────────┴─────────────┘

Guardado: /home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba/submission_202002.csv

Submit v4 ya realizado; se evita duplicarlo.


## 10 — Ensemble opcional con la v3

Busca `pipe_v3_feb2020/submission_x1.0.csv`, genera tres combinaciones y no las sube
salvo que `submit_ensembles=True`. Así se conserva la cuota diaria de Kaggle.


In [11]:
# Ensemble de dos modelos con errores distintos
path_v3 = BUCKET / "pipe_v3_feb2020" / "submission_x1.0.csv"
DIR_ENS = DIR_OUT / "ensembles"
DIR_ENS.mkdir(parents=True, exist_ok=True)

if not path_v3.exists():
    print(f"Todavia no existe {path_v3}. La v4 pura ya quedo generada.")
else:
    v3 = pl.read_csv(path_v3).rename({"tn": "tn_v3"})
    v4 = submit.rename({"tn": "tn_v4"})
    ambos = v3.join(v4, on="product_id", how="inner")
    if ambos.height != oficiales.height:
        raise RuntimeError(f"Ensemble incompleto: {ambos.height}/{oficiales.height} productos")

    for peso_v4 in (0.25, 0.50, 0.75):
        peso_v3 = 1.0 - peso_v4
        ens = ambos.select(
            "product_id",
            (pl.col("tn_v3") * peso_v3 + pl.col("tn_v4") * peso_v4)
              .clip(lower_bound=0.0).alias("tn")
        ).sort("product_id")
        nombre = f"submission_ens_v3_{peso_v3:.2f}_v4_{peso_v4:.2f}.csv"
        ruta = DIR_ENS / nombre
        ens.write_csv(ruta)
        print(f"{nombre}: suma={ens['tn'].sum():,.1f}")

        if PARAM['submit_ensembles']:
            flag = DIR_ENS / f"{nombre}.done"
            if flag.exists():
                print("  ya enviado")
                continue
            ok, salida = kaggle_cli(["competitions", "submit",
                                     "-c", PARAM['kaggle_competition'],
                                     "-f", str(ruta),
                                     "-m", f"ensemble v3={peso_v3:.2f} v4={peso_v4:.2f}"])
            print("  " + ("enviado" if ok else "fallo") + ": " + salida[-200:])
            if ok:
                flag.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


Todavia no existe /home/ds/buckets/b1/pipe_v3_feb2020/submission_x1.0.csv. La v4 pura ya quedo generada.


## 11 — Registro y leaderboard

Una fila por experimento en `exp_residuo/leaderboard_residuo.csv`, con **el WAPE de los
seis esquemas en cada corrida**. Así se ve si el ranking entre esquemas es estable al
cambiar el baseline o las features, que es lo que decide si el hallazgo es real.

In [12]:
resultado = {
    'experimento': EXPERIMENTO,
    'idea': 'residuo sobre baseline: el nivel lo hace el baseline, la desviacion el modelo',
    'granularidad': 'producto-mes',
    'baseline_elegido': BASELINE, 'wape_baselines_val': wape_base,
    'esquema_elegido': ESQUEMA, 'metricas_por_esquema': METRICAS,
    'horizonte': H, 'max_lags': L,
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'n_features': len(FEATURES), 'features': FEATURES,
    'n_trials': len(study.trials) if study else 0,
    'hiperparametros': study.best_params if study else {},
    'pesos_ridge_baseline': {k: round(float(v), 5) for k, v in _co.items()},
    'ridge_alpha': PARAM['ridge_alpha'],
    'n_sin_prediccion': sin_pred,
    'tn_total': float(submit['tn'].sum()),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {'experimento': EXPERIMENTO, 'baseline': BASELINE, 'esquema': ESQUEMA,
        'max_lags': L, 'n_features': len(FEATURES),
        'wape_test_ganador': round(METRICAS[ESQUEMA]['test'], 5),
        'wape_val_ganador': round(METRICAS[ESQUEMA]['val'], 5),
        **{f"test_{e}": round(METRICAS[e]['test'], 5) for e in ESQUEMAS},
        'mejora_vs_baseline_pct': round(100*(_a-_g)/_a, 2),
        'sin_prediccion': sin_pred, 'tn_total': round(float(submit['tn'].sum()), 1)}

path_lb = RUTA_EXP / "leaderboard_residuo.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("wape_test_ganador").write_csv(path_lb)

print(f"Archivos en {DIR_OUT.relative_to(BUCKET)}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_residuo.csv ({nueva.height} experimentos):")
print(nueva.select("baseline", "esquema", "wape_test_ganador",
                   "test_A_baseline", "mejora_vs_baseline_pct"))

Archivos en exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba:
  - diagnostico_extremos_v6.parquet
  - ensembles
  - esquemas.csv
  - panel_cliente_producto_periodo.parquet
  - por_cuartil_de_volumen.csv
  - predicciones_inferencia.parquet
  - predicciones_v5.parquet
  - resultado.json
  - resultado_v5.json
  - resultado_v6.json
  - resumen_panel_denso.csv
  - submission_202002.csv
  - submission_v5_clusters_tweedie.csv
  - submission_v6_nivel_variacion_tendencia.csv
  - submit_v4.done
  - submit_v5.done
  - submit_v6.done

leaderboard_residuo.csv (4 experimentos):
shape: (4, 5)
┌──────────┬──────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ baseline ┆ esquema      ┆ wape_test_ganador ┆ test_A_baseline ┆ mejora_vs_baseline_pct │
│ ---      ┆ ---          ┆ ---               ┆ ---             ┆ ---                    │
│ str      ┆ str          ┆ f64               ┆ f64             ┆ f64            

## Cómo leer el resultado

La tabla de la sección 7 contesta la pregunta directamente. Cuatro desenlaces posibles,
y ninguno es un fracaso:

| Si gana… | Significa |
|---|---|
| **A_baseline** | Ningún modelo le gana a un promedio. Es un resultado fuerte y publicable: en esta serie el ruido domina y la complejidad no compra nada. |
| **C_lineal_nivel** | Se confirma tu observación: la señal es lineal en los niveles recientes, y el GBM está de más. |
| **E_lineal_mas_lgbm** | La hipótesis del notebook: la recta hace el nivel y el árbol agrega algo sobre el residuo. El mejor de los dos mundos. |
| **B** o **F** | El GBM sí puede con el nivel, y lo de la regresión lineal era un problema de tuneo, no estructural. |

Y hay un número que conviene mirar antes de festejar: **la correlación entre la
predicción final y el baseline**, que imprime la sección 8. Si es 0,99, el modelo está
repitiendo el promedio con pasos extra — el WAPE puede mejorar un poco y aun así no
haber aprendido nada.

### Qué variar después

1. **El baseline.** `ma3` contra `lineal` es la comparación que más importa, porque
   `lineal` *es* el promedio con pesos aprendidos.
2. **`max_lags`.** Si la Ridge le pone peso a `tn_lag11` y `tn_lag12`, hay estacionalidad
   anual y conviene subirlo.
3. **`ridge_alpha`.** Con lags muy correlacionados, este número decide si los
   coeficientes salen estables o alternando de signo. Mirá los pesos que imprime la
   sección 4.

## 12 — Clusters causales + LightGBM Tweedie

Cada fila se asigna a un cluster usando sólo información disponible en ese período.
Optuna compara configuraciones sobre febrero de 2018 y febrero de 2019.


In [13]:
# Variables de forma/estado para clusterizar. No se usa clase_tn.
CLUSTER_FEATURES = [c for c in [
    "tn", "tn_lag1", "tn_lag2", "tn_lag3", "tn_lag6", "tn_lag12",
    "tn_ma3", "tn_ma6", "tn_ma12", "tn_dma3", "frac_venta_6",
    "n_clientes", "n_clientes_lag1", "n_clientes_ma3",
    "sh_cat3", "sh_mercado", "idx_tn_mom", "idx_tn_vs_ma3"
] if c in df.columns]


def matriz_cluster(b):
    x = b.select(CLUSTER_FEATURES).fill_null(0.0).to_numpy().astype(np.float64)
    return np.nan_to_num(x, nan=0.0, posinf=10.0, neginf=-10.0)


def params_tweedie(p, semilla):
    q = dict(p)
    q.update(objective="tweedie", metric="mae", max_bin=1023,
             seed=semilla, n_jobs=-1, verbosity=-1,
             deterministic=True, force_row_wise=True)
    return q


def predecir_cluster_tweedie(meses_fit, b_eval, pars, k, semilla=102191):
    fit = bloque(meses_fit)
    xf, xe = matriz_cluster(fit), matriz_cluster(b_eval)
    scaler = StandardScaler().fit(xf)
    zf, ze = scaler.transform(xf), scaler.transform(xe)
    km = KMeans(n_clusters=k, random_state=semilla, n_init=10).fit(zf)
    cf, ce = km.labels_, km.predict(ze)
    pred = np.zeros(b_eval.height, dtype=np.float64)
    bp_fit, bp_eval = fit.to_pandas(), b_eval.to_pandas()

    # Respaldo global para clusters pequeños o sin filas de inferencia.
    global_model = fit_lgbm(fit, "clase_tn", params_tweedie(pars, semilla))
    pred_global = global_model.predict(bp_eval[FEATURES])

    tamanios = {}
    for cl in range(k):
        ix_fit, ix_eval = np.where(cf == cl)[0], np.where(ce == cl)[0]
        tamanios[int(cl)] = {"train": int(len(ix_fit)), "eval": int(len(ix_eval))}
        if not len(ix_eval):
            continue
        if len(ix_fit) < 150 or np.unique(fit["product_id"].to_numpy()[ix_fit]).size < 15:
            pred[ix_eval] = pred_global[ix_eval]
            continue
        sub = fit[ix_fit.tolist()]
        m = fit_lgbm(sub, "clase_tn", params_tweedie(pars, semilla + cl + 1))
        pred[ix_eval] = m.predict(bp_eval.iloc[ix_eval][FEATURES])
    return np.maximum(pred, 0.0), tamanios


def espacio_v5(trial):
    return dict(
        tweedie_variance_power=trial.suggest_float("tweedie_variance_power", 1.05, 1.8),
        num_leaves=trial.suggest_int("num_leaves", 15, 127),
        max_depth=trial.suggest_int("max_depth", 4, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
        n_estimators=trial.suggest_int("n_estimators", 250, 750),
        min_child_samples=trial.suggest_int("min_child_samples", 15, 150),
        subsample=trial.suggest_float("subsample", 0.65, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-6, 3.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-6, 5.0, log=True),
    )


# El ancla global usa en ambos folds los hiperparámetros elegidos por la v4.2.
pred_global_val_v5, _ = predecir_esquema(
    "B_lgbm_nivel", MESES_TRAIN, va, _pars, semilla=PARAM["semilla"])
pred_global_test_v5, _ = predecir_esquema(
    "B_lgbm_nivel", MESES_FIT_TEST, te, _pars, semilla=PARAM["semilla"])
FOLDS_V5 = [
    ("feb2018", MESES_TRAIN, va, pred_global_val_v5),
    ("feb2019", MESES_FIT_TEST, te, pred_global_test_v5),
]

study_v5 = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=PARAM["semilla"]),
    study_name=EXPERIMENTO + "_clusters_tweedie",
    storage=f"sqlite:///{Path.home() / ('optuna_' + EXPERIMENTO + '_clusters.db')}",
    load_if_exists=True,
)


def objective_v5(trial):
    k = trial.suggest_int("k", 3, 7)
    pars = espacio_v5(trial)
    scores = []
    for _, meses_fit, beval, _ in FOLDS_V5:
        pc, _ = predecir_cluster_tweedie(meses_fit, beval, pars, k, PARAM["semilla"])
        scores.append(wape_de(beval, pc))
    trial.set_user_attr("wape_feb2018", scores[0])
    trial.set_user_attr("wape_feb2019", scores[1])
    return float(np.mean(scores))


N_TRIALS_V5 = 25
t0 = time.time()
study_v5.optimize(objective_v5, n_trials=N_TRIALS_V5)
best_v5 = dict(study_v5.best_params)
K_V5 = int(best_v5.pop("k"))
PARS_V5 = params_tweedie(best_v5, PARAM["semilla"])
print(f"{len(study_v5.trials)} trials · k={K_V5} · WAPE medio={study_v5.best_value:.5f} "
      f"· {time.time()-t0:.0f}s")

# Predicciones walk-forward del mejor cluster y búsqueda del ensemble/calibración.
fold_preds = []
for nombre, meses_fit, beval, pglobal in FOLDS_V5:
    pcl, tamanios = predecir_cluster_tweedie(
        meses_fit, beval, PARS_V5, K_V5, PARAM["semilla"])
    fold_preds.append((nombre, beval, np.maximum(pglobal, 0), pcl))
    print(nombre, tamanios)

candidatos = []
for alpha in np.arange(0.0, 1.01, 0.1):  # alpha = peso del modelo cluster
    for mult in np.arange(0.90, 1.101, 0.025):
        sc = []
        for _, beval, pg, pc in fold_preds:
            pred = mult * ((1-alpha)*pg + alpha*pc)
            sc.append(wape_de(beval, pred))
        candidatos.append((float(np.mean(sc)), float(max(sc)), float(alpha), float(mult), sc))

# Primero promedio; el peor fold desempata para favorecer estabilidad.
candidatos.sort(key=lambda z: (z[0], z[1]))
WAPE_CV_V5, _, ALPHA_V5, MULT_V5, SCORES_V5 = candidatos[0]
print(f"ensemble elegido: cluster={ALPHA_V5:.1f}, global={1-ALPHA_V5:.1f}, "
      f"multiplicador={MULT_V5:.3f}")
print(f"WAPE folds: {SCORES_V5} · promedio={WAPE_CV_V5:.5f}")


25 trials · k=6 · WAPE medio=0.25708 · 684s
feb2018 {0: {'train': 1113, 'eval': 17}, 1: {'train': 1496, 'eval': 158}, 2: {'train': 2174, 'eval': 341}, 3: {'train': 59, 'eval': 13}, 4: {'train': 226, 'eval': 27}, 5: {'train': 91, 'eval': 8}}
feb2019 {0: {'train': 1384, 'eval': 28}, 1: {'train': 228, 'eval': 6}, 2: {'train': 5812, 'eval': 435}, 3: {'train': 669, 'eval': 29}, 4: {'train': 3996, 'eval': 149}, 5: {'train': 179, 'eval': 9}}
ensemble elegido: cluster=0.7, global=0.3, multiplicador=0.900
WAPE folds: [0.2221058372905042, 0.1800722406499605] · promedio=0.20109


## 13 — Entrenamiento final, submission y envío


In [14]:
# Entrenamiento final con toda la supervisión disponible.
pred_cluster_semillas = []
tamanios_finales = None
for sem in PARAM["semillas_ensemble"]:
    pc, tamanios_finales = predecir_cluster_tweedie(
        MESES_TODOS, infer, PARS_V5, K_V5, sem)
    pred_cluster_semillas.append(pc)
    print(f"cluster Tweedie semilla {sem} lista")

pred_cluster_final = np.mean(pred_cluster_semillas, axis=0)
pred_v5 = np.maximum(
    MULT_V5 * ((1-ALPHA_V5)*pred_infer_tn + ALPHA_V5*pred_cluster_final),
    PARAM["clip_min"])

pred_final = infer.select("product_id", "periodo", "periodo_objetivo").with_columns(
    pl.Series("pred_global", pred_infer_tn),
    pl.Series("pred_cluster", pred_cluster_final),
    pl.Series("tn_pred", pred_v5),
)
pred_final.write_parquet(DIR_OUT / "predicciones_v5.parquet")

obj_v5 = pred_final.filter(pl.col("periodo_objetivo") == PARAM["periodo_objetivo"])
por_producto_v5 = obj_v5.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
oficiales_v5 = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")
submit_v5 = (oficiales_v5.select("product_id")
             .join(por_producto_v5, on="product_id", how="left")
             .with_columns(pl.col("tn").fill_null(0.0))
             .sort("product_id"))

path_v5 = DIR_OUT / "submission_v5_clusters_tweedie.csv"
submit_v5.write_csv(path_v5)
print(f"guardado: {path_v5}")
print(f"tn total={submit_v5['tn'].sum():,.1f} · productos={submit_v5.height} "
      f"· sin predicción={submit_v5['tn'].null_count()}")

registro_v5 = {
    "k": K_V5, "alpha_cluster": ALPHA_V5, "multiplicador": MULT_V5,
    "wape_folds": SCORES_V5, "wape_cv": WAPE_CV_V5,
    "best_params": study_v5.best_params, "tamanios_finales": tamanios_finales,
    "submission": str(path_v5),
}
with open(DIR_OUT / "resultado_v5.json", "w", encoding="utf-8") as f:
    json.dump(registro_v5, f, indent=2, ensure_ascii=False, default=str)

# Un único envío automático: el ensemble elegido por walk-forward.
SUBMIT_V5 = False
flag_v5 = DIR_OUT / "submit_v5.done"
if not SUBMIT_V5:
    print("SUBMIT_V5=False: CSV generado, no enviado")
elif flag_v5.exists():
    print("v5 ya fue enviada; se evita duplicarla")
else:
    kd = Path.home() / ".kaggle" / "kaggle.json"
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kd); kd.chmod(0o600); break
    if not kd.exists():
        print("Sin credenciales Kaggle; CSV generado")
    else:
        kd.chmod(0o600)
        msg = (f"v5 clusters Tweedie k={K_V5} blend={ALPHA_V5:.1f} "
               f"mult={MULT_V5:.3f} cv={WAPE_CV_V5:.5f}")
        ok, salida = kaggle_cli(["competitions", "submit",
                                 "-c", PARAM["kaggle_competition"],
                                 "-f", str(path_v5), "-m", msg])
        print(salida)
        if ok:
            flag_v5.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
            print("Submit v5 enviado")
        else:
            print("No se pudo enviar; CSV generado correctamente")


cluster Tweedie semilla 109903 lista
cluster Tweedie semilla 109927 lista
guardado: /home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba/submission_v5_clusters_tweedie.csv
tn total=26,066.3 · productos=780 · sin predicción=0
SUBMIT_V5=False: CSV generado, no enviado


## 14 — Variación logarítmica y tendencia extrapolable

Todas las pendientes terminan en el mes de origen. No se calculan centradas ni
usan observaciones posteriores.


In [15]:
# Features causales adicionales: sólo mes t y anteriores.
df6 = df.with_columns(
    ((pl.col("tn") - pl.col("tn_lag2")) / 2).alias("pend_3"),
    ((pl.col("tn") - pl.col("tn_lag6")) / 6).alias("pend_6"),
    ((pl.col("tn") - pl.col("tn_lag12")) / 12).alias("pend_12"),
    pl.col("tn").rolling_min(12).over("product_id").alias("min_12"),
    pl.col("tn").rolling_max(12).over("product_id").alias("max_12"),
).with_columns(
    (pl.col("pend_3") - pl.col("pend_6")).alias("aceleracion_3_6"),
    (pl.col("tn") - pl.col("min_12")).alias("dist_min_12"),
    (pl.col("max_12") - pl.col("tn")).alias("dist_max_12"),
)

EXTRA6 = ["pend_3", "pend_6", "pend_12", "aceleracion_3_6",
          "min_12", "max_12", "dist_min_12", "dist_max_12"]
FEATURES6 = FEATURES + EXTRA6
sup6 = df6.filter(pl.col("clase_tn").is_not_null())


def enriquecer6(b):
    """Recupera las columnas v6 preservando exactamente el orden del bloque original."""
    claves = b.select("product_id", "periodo").with_row_index("_orden6")
    return (claves.join(df6, on=["product_id", "periodo"], how="left")
                  .sort("_orden6").drop("_orden6"))


def bloque6(meses):
    return sup6.filter(pl.col("periodo").is_in(meses))


def base_robusta(b):
    # Combina nivel actual y media corta; no fija techo ni piso histórico.
    return np.maximum(0.55*b["tn"].fill_null(0).to_numpy()
                      + 0.30*b["tn_ma3"].fill_null(0).to_numpy()
                      + 0.15*b["tn_ma6"].fill_null(0).to_numpy(), 0.0)


def fit_predict_variacion(meses_fit, beval, semilla=102191):
    fit = bloque6(meses_fit)
    beval = enriquecer6(beval)
    escala_fit = base_robusta(fit) + 0.05
    y = np.log((fit["clase_tn"].to_numpy() + 0.05) / escala_fit)
    # Evita que unos pocos ceros/picos dominen, sin limitar la predicción al máximo histórico.
    y = np.clip(y, -3.5, 3.5)
    pars = dict(PARS_V5)
    pars.update(objective="huber", metric="mae", seed=semilla,
                n_estimators=min(int(pars.get("n_estimators", 500)), 650))
    pars.pop("tweedie_variance_power", None)
    m = lgb.LGBMRegressor(**pars)
    pf, pe = fit.to_pandas(), beval.to_pandas()
    m.fit(pf[FEATURES6], y, categorical_feature=CATS)
    r = np.clip(m.predict(pe[FEATURES6]), -2.5, 2.5)
    return np.maximum((base_robusta(beval)+0.05)*np.exp(r)-0.05, 0.0)


TREND6 = ["tn", "tn_lag1", "tn_lag2", "tn_lag3", "tn_lag6", "tn_lag12",
          "tn_ma3", "tn_ma6", "tn_ma12"] + EXTRA6


def matriz_tendencia(b):
    x = b.select(TREND6).fill_null(0.0).to_numpy().astype(float)
    nivel = np.maximum(b["tn_ma6"].fill_null(0).to_numpy(), 0.1)
    return np.nan_to_num(x / nivel[:, None]), nivel


def fit_predict_tendencia(meses_fit, beval):
    fit = bloque6(meses_fit)
    beval = enriquecer6(beval)
    xf, nf = matriz_tendencia(fit)
    xe, ne = matriz_tendencia(beval)
    # Delta relativo: Ridge puede extrapolar fuera de los valores observados por los árboles.
    yf = (fit["clase_tn"].to_numpy() - base_robusta(fit)) / nf
    scaler = StandardScaler().fit(xf)
    r = Ridge(alpha=8.0).fit(scaler.transform(xf), np.clip(yf, -5, 5))
    delta = np.clip(r.predict(scaler.transform(xe)), -2.0, 3.0)
    return np.maximum(base_robusta(beval) + ne*delta, 0.0)


# Reconstruye la predicción v5 de cada fold con los pesos ya elegidos.
folds6 = []
for (nombre, meses_fit, beval, pg), (_, _, _, pc) in zip(FOLDS_V5, fold_preds):
    pv5 = np.maximum(MULT_V5*((1-ALPHA_V5)*pg + ALPHA_V5*pc), 0.0)
    pvar = fit_predict_variacion(meses_fit, beval, PARAM["semilla"])
    ptr = fit_predict_tendencia(meses_fit, beval)
    folds6.append((nombre, beval, pv5, pvar, ptr))
    print(nombre, "WAPE v5/variación/tendencia:",
          round(wape_de(beval, pv5),5), round(wape_de(beval,pvar),5),
          round(wape_de(beval,ptr),5))

# Pesos simples y estables. Penalizamos diferencias fuertes entre los dos febreros.
candidatos6 = []
for wv5 in np.arange(0.4, 1.01, 0.1):
    for wvar in np.arange(0.0, 1.01-wv5, 0.1):
        wtrend = 1.0-wv5-wvar
        for mult in np.arange(0.925, 1.076, 0.025):
            sc=[]
            for _, be, p5, pv, pt in folds6:
                pp = mult*(wv5*p5 + wvar*pv + wtrend*pt)
                sc.append(wape_de(be,pp))
            criterio=float(np.mean(sc)+0.25*abs(sc[0]-sc[1]))
            candidatos6.append((criterio,float(np.mean(sc)),wv5,wvar,wtrend,float(mult),sc))
candidatos6.sort(key=lambda z:z[0])
CRIT6,WAPE6,W5_6,WVAR6,WTREND6,MULT6,SCORES6=candidatos6[0]
print(f"pesos v6: v5={W5_6:.1f}, variación={WVAR6:.1f}, tendencia={WTREND6:.1f}, "
      f"mult={MULT6:.3f}")
print(f"WAPE folds={SCORES6}, promedio={WAPE6:.5f}, criterio estable={CRIT6:.5f}")


feb2018 WAPE v5/variación/tendencia: 0.22211 0.37335 0.30041
feb2019 WAPE v5/variación/tendencia: 0.18006 0.22531 0.20844
pesos v6: v5=0.8, variación=0.0, tendencia=0.2, mult=0.925
WAPE folds=[0.2019218008927428, 0.18198118705801003], promedio=0.19195, criterio estable=0.19694


## 15 — Predicción final, diagnóstico de extremos y submit


In [16]:
# Dos semillas para la rama multiplicativa; Ridge es determinista.
pvars=[]
infer6=enriquecer6(infer)
for sem in PARAM["semillas_ensemble"]:
    pvars.append(fit_predict_variacion(MESES_TODOS, infer, sem))
pred_var6=np.mean(pvars,axis=0)
pred_trend6=fit_predict_tendencia(MESES_TODOS,infer)
pred6=np.maximum(MULT6*(W5_6*pred_v5 + WVAR6*pred_var6 + WTREND6*pred_trend6),0.0)

diag6=infer6.select("product_id","periodo","periodo_objetivo","max_12","min_12").with_columns(
    pl.Series("pred_v5",pred_v5),pl.Series("pred_variacion",pred_var6),
    pl.Series("pred_tendencia",pred_trend6),pl.Series("tn_pred",pred6),
).with_columns(
    (pl.col("tn_pred")>pl.col("max_12")).alias("supera_max_12"),
    (pl.col("tn_pred")<pl.col("min_12")).alias("debajo_min_12"),
)
diag6.write_parquet(DIR_OUT/"diagnostico_extremos_v6.parquet")

obj6=diag6.filter(pl.col("periodo_objetivo")==PARAM["periodo_objetivo"])
porprod6=obj6.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
of6=pl.read_csv(DIR_RAW/"product_id_apredecir201912.txt",separator="\t")
sub6=(of6.select("product_id").join(porprod6,on="product_id",how="left")
      .with_columns(pl.col("tn").fill_null(0.0)).sort("product_id"))
path6=DIR_OUT/"submission_v6_nivel_variacion_tendencia.csv"
sub6.write_csv(path6)
print(f"superan máximo 12m: {int(obj6['supera_max_12'].sum())} · "
      f"debajo mínimo 12m: {int(obj6['debajo_min_12'].sum())}")
print(f"tn total={sub6['tn'].sum():,.1f} · archivo={path6}")

res6={"pesos":{"v5":W5_6,"variacion":WVAR6,"tendencia":WTREND6},
      "multiplicador":MULT6,"wape_folds":SCORES6,"wape_promedio":WAPE6,
      "supera_max_12":int(obj6["supera_max_12"].sum()),
      "debajo_min_12":int(obj6["debajo_min_12"].sum())}
with open(DIR_OUT/"resultado_v6.json","w",encoding="utf-8") as f:
    json.dump(res6,f,indent=2,ensure_ascii=False)

SUBMIT_V6=False
flag6=DIR_OUT/"submit_v6.done"
if SUBMIT_V6 and not flag6.exists():
    kd=Path.home()/".kaggle"/"kaggle.json"
    kd.parent.mkdir(parents=True,exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET/"kaggle.json",BUCKET/"kaggle"/"kaggle.json"):
            if cand.exists(): shutil.copy(cand,kd); kd.chmod(0o600); break
    if kd.exists():
        kd.chmod(0o600)
        msg=(f"v6 nivel-var-tend w={W5_6:.1f}/{WVAR6:.1f}/{WTREND6:.1f} "
             f"mult={MULT6:.3f} cv={WAPE6:.5f}")
        ok,salida=kaggle_cli(["competitions","submit","-c",PARAM["kaggle_competition"],
                              "-f",str(path6),"-m",msg])
        print(salida)
        if ok: flag6.write_text(time.strftime("%Y-%m-%d %H:%M:%S")); print("Submit v6 enviado")
    else: print("Sin credenciales Kaggle; CSV generado")
elif flag6.exists(): print("v6 ya enviada; se evita duplicarla")


superan máximo 12m: 2 · debajo mínimo 12m: 17
tn total=24,437.8 · archivo=/home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v5_clusters_tweedie_ba/submission_v6_nivel_variacion_tendencia.csv
v6 ya enviada; se evita duplicarla


## 16 — Régimen 07 y especialistas por cluster


In [17]:
# Ventanas exactas de 07_Residuo_sobre_baseline.
MESES_07_TRAIN=[p for p in periodos_sup if 201701<=p<=201905]
MESES_07_VAL=[p for p in [201907,201908] if p in periodos_sup]
MESES_07_TEST=[p for p in [201910] if p in periodos_sup]
val07=bloque(MESES_07_VAL); test07=bloque(MESES_07_TEST)

# Optuna del modelo 07 al nivel, seleccionado sólo con julio/agosto 2019.
study07=optuna.create_study(direction='minimize',
 sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
 study_name=EXPERIMENTO+'_v8_07',
 storage=f"sqlite:///{Path.home()/('optuna_'+EXPERIMENTO+'_v8_07.db')}",load_if_exists=True)
def obj07(trial):
 p=espacio(trial)
 pred,_=predecir_esquema('B_lgbm_nivel',MESES_07_TRAIN,val07,p,PARAM['semilla'])
 return wape_de(val07,pred)
study07.optimize(obj07,n_trials=30)
PARS07={**espacio(optuna.trial.FixedTrial(study07.best_params))}
print(f"07: {len(study07.trials)} trials, WAPE val={study07.best_value:.5f}")

# Predicciones out-of-time de ambos candidatos.
p07_val,_=predecir_esquema('B_lgbm_nivel',MESES_07_TRAIN,val07,PARS07,PARAM['semilla'])
fit07_test=sorted(set(MESES_07_TRAIN+MESES_07_VAL))
p07_test,_=predecir_esquema('B_lgbm_nivel',fit07_test,test07,PARS07,PARAM['semilla'])
p6_val=fit_predict_variacion(MESES_07_TRAIN,val07,PARAM['semilla'])
p6_test=fit_predict_variacion(fit07_test,test07,PARAM['semilla'])

# Clusters estables por producto, construidos con información disponible a mayo-2019.
perfil_cols=[c for c in ['tn','tn_lag1','tn_lag2','tn_lag3','tn_lag6','tn_lag12',
 'tn_ma3','tn_ma6','tn_ma12','frac_venta_6','n_clientes','sh_cat3','sh_mercado'] if c in df.columns]
perfil=(df.filter(pl.col('periodo')==201905).select(['product_id']+perfil_cols)
        .fill_null(0.0).sort('product_id'))
xp=perfil.select(perfil_cols).to_numpy().astype(float)
sc8=StandardScaler().fit(xp)
km8=KMeans(n_clusters=5,random_state=PARAM['semilla'],n_init=20).fit(sc8.transform(xp))
map_cluster=dict(zip(perfil['product_id'].to_list(),km8.labels_.tolist()))

def cluster_de(b):
 return np.array([map_cluster.get(int(x),-1) for x in b['product_id'].to_list()])
cv,ct=cluster_de(val07),cluster_de(test07)

# Elige peso de v6 por cluster. Shrink hacia blend global si el cluster tiene poco volumen.
fold8=[(val07,p07_val,p6_val,cv),(test07,p07_test,p6_test,ct)]
pesos_grid=np.arange(0,1.01,0.1)
def score_peso(w,cluster=None):
 ss=[]; volumen=0.0; n=0
 for b,a,z,c in fold8:
  mask=np.ones(len(a),dtype=bool) if cluster is None else c==cluster
  if not mask.any(): continue
  real=b['clase_tn'].to_numpy()[mask]; pred=(1-w)*a[mask]+w*z[mask]
  ss.append(wape(real,pred,b['product_id'].to_numpy()[mask])); volumen+=float(real.sum()); n+=mask.sum()
 return (float(np.mean(ss)) if ss else np.inf),volumen,n

globales=[(score_peso(w)[0],w) for w in pesos_grid]
WGLOBAL=min(globales)[1]
decision=[]
for cl in range(5):
 cand=[(score_peso(w,cl)[0],w,score_peso(w,cl)[1],score_peso(w,cl)[2]) for w in pesos_grid]
 err,wraw,vol,n=min(cand)
 # Regularización: con poca evidencia, acerca el peso al global.
 confianza=min(1.0,n/120.0,vol/500.0)
 w=round((confianza*wraw+(1-confianza)*WGLOBAL)*10)/10
 decision.append({'cluster':cl,'peso_v6':w,'peso_07':1-w,'wape_cv':err,
                  'n':int(n),'volumen':round(vol,2),'confianza':round(confianza,3)})
tabla8=pl.DataFrame(decision)
print(f"peso global v6={WGLOBAL:.1f}"); print(tabla8)

# Entrenamiento 07 final con toda fila cuyo target ya existe; v6 ya fue entrenada arriba.
p07_final,_=predecir_esquema('B_lgbm_nivel',MESES_TODOS,infer,PARS07,PARAM['semilla'])
clusters_final=cluster_de(infer)
wp=np.array([decision[c]['peso_v6'] if c>=0 else WGLOBAL for c in clusters_final])
pred8=np.maximum((1-wp)*p07_final+wp*pred6,0.0)

# Control externo: CSV original 07. Sólo verifica; no elige pesos ni mira targets.
candidatos07=[BUCKET/'submission_ultima (1).csv',BUCKET/'submission_ultima.csv',
              DIR_RAW/'submission_ultima (1).csv']
ref07=next((p for p in candidatos07 if p.exists()),None)
control07=None
if ref07:
 r07=pl.read_csv(ref07).sort('product_id')
 interno=(infer.select('product_id','periodo_objetivo').with_columns(pl.Series('tn',p07_final))
          .filter(pl.col('periodo_objetivo')==PARAM['periodo_objetivo'])
          .group_by('product_id').agg(pl.col('tn').sum()).sort('product_id'))
 comp=r07.join(interno,on='product_id',how='inner',suffix='_interno')
 control07=float((comp['tn']-comp['tn_interno']).abs().mean())
 print(f"control reproducción 07: MAE={control07:.6f} tn")
else: print('CSV 07 no encontrado en bucket: se omite control externo.')

pred8df=infer.select('product_id','periodo','periodo_objetivo').with_columns(
 pl.Series('cluster',clusters_final),pl.Series('peso_v6',wp),
 pl.Series('pred_07',p07_final),pl.Series('pred_v6',pred6),pl.Series('tn_pred',pred8))
pred8df.write_parquet(DIR_OUT/'diagnostico_v8.parquet')
obj8=pred8df.filter(pl.col('periodo_objetivo')==PARAM['periodo_objetivo'])
por8=obj8.group_by('product_id').agg(pl.col('tn_pred').sum().alias('tn'))
of8=pl.read_csv(DIR_RAW/'product_id_apredecir201912.txt',separator='\t')
sub8=(of8.select('product_id').join(por8,on='product_id',how='left')
      .with_columns(pl.col('tn').fill_null(0.0)).sort('product_id'))
path8=DIR_OUT/'submission_v8_07_v6_especialistas_cluster.csv'; sub8.write_csv(path8)
tabla8.write_csv(DIR_OUT/'decision_modelo_por_cluster.csv')
with open(DIR_OUT/'resultado_v8.json','w',encoding='utf-8') as f:
 json.dump({'peso_global_v6':WGLOBAL,'clusters':decision,'control_mae_07':control07},f,indent=2)
print(f"submission={path8} · tn total={sub8['tn'].sum():,.1f}")

SUBMIT_V8=True; flag8=DIR_OUT/'submit_v8.done'
if SUBMIT_V8 and not flag8.exists():
 kd=Path.home()/'.kaggle'/'kaggle.json'; kd.parent.mkdir(parents=True,exist_ok=True)
 if not kd.exists():
  for cand in (BUCKET/'kaggle.json',BUCKET/'kaggle'/'kaggle.json'):
   if cand.exists(): shutil.copy(cand,kd); kd.chmod(0o600); break
 if kd.exists():
  kd.chmod(0o600); msg=f"v8 07+v6 especialistas cluster wglobal={WGLOBAL:.1f}"
  ok,salida=kaggle_cli(['competitions','submit','-c',PARAM['kaggle_competition'],
                        '-f',str(path8),'-m',msg]); print(salida)
  if ok: flag8.write_text(time.strftime('%Y-%m-%d %H:%M:%S')); print('Submit v8 enviado')
elif flag8.exists(): print('v8 ya enviada; se evita duplicarla')


07: 30 trials, WAPE val=0.19519
peso global v6=0.9
shape: (5, 7)
┌─────────┬─────────┬─────────┬──────────┬──────┬──────────┬───────────┐
│ cluster ┆ peso_v6 ┆ peso_07 ┆ wape_cv  ┆ n    ┆ volumen  ┆ confianza │
│ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---  ┆ ---      ┆ ---       │
│ i64     ┆ f64     ┆ f64     ┆ f64      ┆ i64  ┆ f64      ┆ f64       │
╞═════════╪═════════╪═════════╪══════════╪══════╪══════════╪═══════════╡
│ 0       ┆ 1.0     ┆ 0.0     ┆ 0.295    ┆ 183  ┆ 1566.22  ┆ 1.0       │
│ 1       ┆ 0.8     ┆ 0.2     ┆ 0.214335 ┆ 258  ┆ 34048.08 ┆ 1.0       │
│ 2       ┆ 1.0     ┆ 0.0     ┆ 0.246946 ┆ 1656 ┆ 21843.87 ┆ 1.0       │
│ 3       ┆ 0.9     ┆ 0.1     ┆ 0.133211 ┆ 6    ┆ 8883.23  ┆ 0.05      │
│ 4       ┆ 0.6     ┆ 0.4     ┆ 0.166477 ┆ 51   ┆ 23669.31 ┆ 0.425     │
└─────────┴─────────┴─────────┴──────────┴──────┴──────────┴───────────┘
control reproducción 07: MAE=3.549100 tn
submission=/home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_v